# Product Page Html Collector

Collects product-detail page HTML for each ASIN with Playwright. Tracks status per ASIN (success / fail / blocked), never retries a blocked ASIN, and skips pages already collected.

**Setup.** Data paths point to `/content/drive/MyDrive/YOUR_PROJECT_FOLDER/...` — change `YOUR_PROJECT_FOLDER` to your own folder before running. API keys are read from Colab Secrets / environment variables, never hard-coded.

> Code comments and console messages are in Korean (original working language). See the [folder README](README.md) for the pipeline overview and the [disclaimer](../README.md#disclaimer).


> ⚠️ **Read before running.** This notebook uses Playwright to open Amazon product pages automatically. Amazon's Conditions of Use restrict robots and data-extraction tools.
>
> It is shared to show the collection and parsing method, not as a tool to run against amazon.com. Do not run it against the live site without that site's permission. To follow the downstream steps, use the synthetic files in [`sample-data/`](../sample-data/).

In [ ]:
# ================================================================================
# Amazon 상품 상세페이지 HTML 수집기
# v5 - BLOCKED 판별 + Progress 안정화 + 기존 HTML 상태 관리
#
# 핵심
#   1. Excel의 ASIN 기준 수집
#   2. 정상 HTML만 SUCCESS로 인정
#   3. FAIL / UNKNOWN / BLOCKED는 재수집 가능
#   4. Amazon 차단 페이지는 _BLOCKED.html로 저장
#   5. BLOCKED ASIN은 동일 ASIN에서 재시도하지 않음
#   6. Progress는 매 상품마다 즉시 갱신
#   7. 기존 SUCCESS HTML은 수집 대상에서 제외
# ================================================================================


# ================================================================================
# 0. 기본 라이브러리
# ================================================================================

import os
import re
import json
import time
import random
import shutil
import zipfile
import asyncio
from datetime import datetime

import pandas as pd
import numpy as np

from google.colab import drive, files

from playwright.async_api import async_playwright


# ================================================================================
# 1. Google Drive 연결
# ================================================================================

print("=" * 80)
print("🔵 Google Drive 연결")
print("=" * 80)

drive.mount("/content/drive", force_remount=False)


# ================================================================================
# 2. 경로 설정
# ================================================================================

BASE_DIR = (
    "/content/drive/MyDrive/"
    "YOUR_PROJECT_FOLDER/amazon_data/amazon_output"
)

HTML_DIR = os.path.join(BASE_DIR, "html")

SCREENSHOT_DIR = os.path.join(
    BASE_DIR,
    "screenshots"
)

PROGRESS_FILE = os.path.join(
    BASE_DIR,
    "collection_progress.csv"
)

RESULT_FILE = os.path.join(
    BASE_DIR,
    "Amazon_productpage_HTML_crawling.xlsx"
)

ZIP_BASE = os.path.join(
    BASE_DIR,
    "Amazon_productpage_HTML"
)


# ================================================================================
# 3. 디렉터리 생성
# ================================================================================

os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(HTML_DIR, exist_ok=True)
os.makedirs(SCREENSHOT_DIR, exist_ok=True)


print(f"BASE_DIR       : {BASE_DIR}")
print(f"HTML_DIR       : {HTML_DIR}")
print(f"SCREENSHOT_DIR : {SCREENSHOT_DIR}")
print(f"PROGRESS_FILE  : {PROGRESS_FILE}")


# ================================================================================
# 4. Excel 업로드
# ================================================================================

print("\n" + "=" * 80)
print("📤 Amazon ASIN Excel 업로드")
print("=" * 80)

uploaded = files.upload()

if not uploaded:
    raise RuntimeError("❌ Excel 파일이 업로드되지 않았습니다.")

uploaded_filename = list(uploaded.keys())[0]

print(f"\n📄 업로드 파일: {uploaded_filename}")


# ================================================================================
# 5. Excel 읽기
# ================================================================================

input_path = os.path.join(
    "/content",
    uploaded_filename
)

df = pd.read_excel(input_path)

print(f"\n📊 데이터 행 수: {len(df):,}")
print(f"📋 컬럼 수: {len(df.columns):,}")

print("\n컬럼:")
print(df.columns.tolist())


# ================================================================================
# 6. ASIN 정리
# ================================================================================

if "ASIN" not in df.columns:
    raise ValueError(
        "❌ Excel에 'ASIN' 컬럼이 없습니다."
    )


def clean_asin(value):

    if pd.isna(value):
        return ""

    value = str(value).strip().upper()

    # 혹시 URL이 들어온 경우
    m = re.search(
        r"/dp/([A-Z0-9]{10})",
        value,
        flags=re.I
    )

    if m:
        return m.group(1)

    # ASIN만 들어온 경우
    m = re.search(
        r"\b([A-Z0-9]{10})\b",
        value,
        flags=re.I
    )

    if m:
        return m.group(1)

    return ""


df["ASIN"] = df["ASIN"].apply(clean_asin)

valid_mask = df["ASIN"].str.len() == 10

valid_df = df.loc[valid_mask].copy()

invalid_df = df.loc[~valid_mask].copy()

print("\n" + "=" * 80)
print("🔎 ASIN 확인")
print("=" * 80)

print(f"✅ 유효 ASIN: {len(valid_df):,}개")
print(f"❌ ASIN 추출 실패: {len(invalid_df):,}개")


# ================================================================================
# 7. 중복 ASIN 확인
# ================================================================================

duplicate_count = (
    valid_df["ASIN"]
    .duplicated()
    .sum()
)

unique_asin_count = (
    valid_df["ASIN"]
    .nunique()
)

print(f"🔢 고유 ASIN: {unique_asin_count:,}개")
print(f"♻️ 중복 ASIN 행: {duplicate_count:,}개")


# ================================================================================
# 8. HTML 상태 판별 기준
# ================================================================================

PRODUCT_MARKERS = [

    "productTitle",
    "feature-bullets",
    "productDetails",
    "detailBullets",
    "productOverview",
    "priceblock",
    "corePrice",
    "buybox",
    "dp-container",
    "product-detail",
    "aod-price",

]


# ------------------------------------------------------------------------------
# Amazon 차단 페이지
# ------------------------------------------------------------------------------

BLOCK_MARKERS = [

    # CAPTCHA / Robot
    "captcha",
    "robot check",
    "robotcheck",
    "enter the characters you see below",
    "sorry, we just need to make sure you're not a robot",
    "sorry, we just need to make sure you're not a robot. please solve this captcha",
    "automated access to amazon data",
    "type the characters you see in this image",

    # Amazon 오류/차단
    "something went wrong",

    # 현재 확인된 Amazon AI-agent 차단 문구
    "continued access by an unauthorized ai agent",
    "violates amazon's conditions of use",

]


# ------------------------------------------------------------------------------
# 일반 오류 페이지
# ------------------------------------------------------------------------------

ERROR_MARKERS = [

    "page not found",
    "looking for something?",
    "sorry! we couldn't find that page",
    "we couldn't find that page",

]


# ================================================================================
# 9. HTML 분석 함수
# ================================================================================

def analyze_html(html):

    if not html:
        return (
            "FAIL",
            "empty_html"
        )

    lower = html.lower()

    html_size = len(html)


    # --------------------------------------------------------------------------
    # ① BLOCKED 확인
    # --------------------------------------------------------------------------

    matched_block = []

    for marker in BLOCK_MARKERS:

        if marker.lower() in lower:

            matched_block.append(marker)


    if matched_block:

        return (
            "BLOCKED",
            "matched=" + " | ".join(
                matched_block[:5]
            )
        )


    # --------------------------------------------------------------------------
    # ② Product 페이지 확인
    # --------------------------------------------------------------------------

    matched_product = []

    for marker in PRODUCT_MARKERS:

        if marker.lower() in lower:

            matched_product.append(marker)


    marker_count = len(matched_product)


    # 정상 Product 페이지는 최소 2개 marker
    if marker_count >= 2:

        return (
            "SUCCESS",
            f"markers={marker_count}, size={html_size}"
        )


    # --------------------------------------------------------------------------
    # ③ 일반 오류 페이지 확인
    # --------------------------------------------------------------------------

    matched_error = []

    for marker in ERROR_MARKERS:

        if marker.lower() in lower:

            matched_error.append(marker)


    if matched_error:

        return (
            "FAIL",
            "error_page=" + " | ".join(
                matched_error[:5]
            )
        )


    # --------------------------------------------------------------------------
    # ④ 너무 작은 HTML
    # --------------------------------------------------------------------------

    if html_size < 10000:

        return (
            "FAIL",
            f"html_too_small={html_size}, markers={marker_count}"
        )


    # --------------------------------------------------------------------------
    # ⑤ 애매한 페이지
    # --------------------------------------------------------------------------

    return (
        "UNKNOWN",
        f"markers={marker_count}, size={html_size}"
    )


# ================================================================================
# 10. HTML 파일 경로
# ================================================================================

def normal_html_path(asin):

    return os.path.join(
        HTML_DIR,
        f"{asin}.html"
    )


def blocked_html_path(asin):

    return os.path.join(
        HTML_DIR,
        f"{asin}_BLOCKED.html"
    )


# ================================================================================
# 11. 기존 HTML 상태 확인
# ================================================================================

def get_html_status(asin):

    normal_path = normal_html_path(asin)

    blocked_path = blocked_html_path(asin)


    # --------------------------------------------------------------------------
    # 정상 HTML
    # --------------------------------------------------------------------------

    if os.path.exists(normal_path):

        try:

            with open(
                normal_path,
                "r",
                encoding="utf-8",
                errors="ignore"
            ) as f:

                html = f.read()


            status, reason = analyze_html(html)

            return (
                status,
                reason,
                normal_path
            )


        except Exception as e:

            return (
                "FAIL",
                f"read_error={e}",
                normal_path
            )


    # --------------------------------------------------------------------------
    # BLOCKED HTML
    # --------------------------------------------------------------------------

    if os.path.exists(blocked_path):

        try:

            with open(
                blocked_path,
                "r",
                encoding="utf-8",
                errors="ignore"
            ) as f:

                html = f.read()


            status, reason = analyze_html(html)


            # 파일명 자체가 BLOCKED이므로
            # 분석 결과가 애매해도 BLOCKED로 취급
            if status != "SUCCESS":

                return (
                    "BLOCKED",
                    reason,
                    blocked_path
                )


            return (
                status,
                reason,
                blocked_path
            )


        except Exception as e:

            return (
                "BLOCKED",
                f"read_error={e}",
                blocked_path
            )


    # --------------------------------------------------------------------------
    # 파일 없음
    # --------------------------------------------------------------------------

    return (
        "NONE",
        "",
        None
    )


# ================================================================================
# 12. 기존 HTML 상태 전체 확인
# ================================================================================

print("\n" + "=" * 80)
print("♻️ 기존 HTML 상태 확인")
print("=" * 80)


existing_success = 0
existing_retry = 0

existing_status_list = []


for asin in valid_df["ASIN"].drop_duplicates():

    status, reason, path = get_html_status(asin)

    existing_status_list.append(
        {
            "ASIN": asin,
            "STATUS": status,
            "REASON": reason,
            "PATH": path,
        }
    )

    if status == "SUCCESS":

        existing_success += 1

    elif status in [
        "FAIL",
        "UNKNOWN",
        "BLOCKED"
    ]:

        existing_retry += 1


print(f"✅ 기존 정상 HTML: {existing_success:,}개")
print(f"🔄 재수집 대상 기존 파일: {existing_retry:,}개")


# ================================================================================
# 13. 이번 실행 수집 대상
#
# 정상 SUCCESS만 제외
# FAIL / UNKNOWN / BLOCKED는 재수집 대상
# ================================================================================

success_asins = {

    x["ASIN"]

    for x in existing_status_list

    if x["STATUS"] == "SUCCESS"

}


remaining_df = valid_df[
    ~valid_df["ASIN"].isin(success_asins)
].copy()


print(
    f"\n🚀 이번 실행 수집 대상: "
    f"{len(remaining_df):,}개"
)


# ================================================================================
# 14. Progress 읽기
# ================================================================================

print("\n" + "=" * 80)
print("📋 Progress 확인")
print("=" * 80)


if os.path.exists(PROGRESS_FILE):

    try:

        progress_df = pd.read_csv(
            PROGRESS_FILE,
            dtype=str
        )

        print(
            f"✅ 기존 Progress: "
            f"{len(progress_df):,}건"
        )

    except Exception as e:

        print(
            f"⚠️ Progress 읽기 실패: {e}"
        )

        progress_df = pd.DataFrame()

else:

    progress_df = pd.DataFrame()

    print("ℹ️ 기존 Progress 없음")


# ================================================================================
# 15. Progress 컬럼
# ================================================================================

PROGRESS_COLUMNS = [

    "ASIN",
    "STATUS",
    "HTTP_STATUS",
    "REASON",
    "HTML_FILE",
    "수집시간",

]


# ================================================================================
# 16. Progress 저장 함수
# ================================================================================

def save_progress(
    progress_df,
    current_result
):

    # current_result는 dict 또는 list 모두 허용
    if isinstance(
        current_result,
        dict
    ):

        current_df = pd.DataFrame(
            [current_result]
        )

    else:

        current_df = pd.DataFrame(
            current_result
        )


    # 빈 데이터 방지
    if current_df.empty:

        return progress_df


    # 필요한 컬럼 보정
    for col in PROGRESS_COLUMNS:

        if col not in progress_df.columns:

            progress_df[col] = ""


        if col not in current_df.columns:

            current_df[col] = ""


    # 컬럼 순서
    progress_df = progress_df[
        PROGRESS_COLUMNS
    ]

    current_df = current_df[
        PROGRESS_COLUMNS
    ]


    # 기존 + 현재 결과
    combined = pd.concat(
        [
            progress_df,
            current_df
        ],
        ignore_index=True
    )


    # ASIN 기준 최신 결과 유지
    combined = (
        combined
        .drop_duplicates(
            subset=["ASIN"],
            keep="last"
        )
        .reset_index(drop=True)
    )


    # 저장
    combined.to_csv(
        PROGRESS_FILE,
        index=False,
        encoding="utf-8-sig"
    )


    return combined


# ================================================================================
# 17. Playwright 브라우저 설정
# ================================================================================

playwright = None
browser = None
context = None
page = None


# ================================================================================
# 18. 브라우저 시작
# ================================================================================

async def start_browser():

    global playwright
    global browser
    global context
    global page


    # 기존 브라우저 종료
    try:

        if browser:

            await browser.close()

    except:

        pass


    browser = None
    context = None
    page = None


    # Playwright 최초 시작
    if playwright is None:

        playwright = await async_playwright().start()


    print("\n🔄 Chromium 재시작")


    browser = await playwright.chromium.launch(

        headless=True,

        args=[

            "--no-sandbox",
            "--disable-dev-shm-usage",
            "--disable-gpu",
            "--disable-blink-features=AutomationControlled",

        ]

    )


    context = await browser.new_context(

        viewport={
            "width": 1366,
            "height": 900
        },

        locale="en-US",

        timezone_id="America/Los_Angeles",

        user_agent=(
            "Mozilla/5.0 "
            "(Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 "
            "(KHTML, like Gecko) "
            "Chrome/153.0.0.0 "
            "Safari/537.36"
        ),

    )


    page = await context.new_page()


    # 페이지 로딩 timeout
    page.set_default_timeout(
        30000
    )

    page.set_default_navigation_timeout(
        30000
    )


# ================================================================================
# 19. 브라우저 종료
# ================================================================================

async def close_browser():

    global browser
    global context
    global page
    global playwright


    try:

        if browser:

            await browser.close()

    except Exception as e:

        print(
            f"⚠️ browser close 오류: {e}"
        )


    browser = None
    context = None
    page = None


    # Playwright 자체는 마지막에 종료
    # 매번 종료하면 재시작이 불편하므로 유지


# ================================================================================
# 20. HTML 저장
# ================================================================================

def save_html(
    asin,
    html,
    status
):

    normal_path = normal_html_path(asin)

    blocked_path = blocked_html_path(asin)


    # --------------------------------------------------------------------------
    # SUCCESS
    # --------------------------------------------------------------------------

    if status == "SUCCESS":

        with open(
            normal_path,
            "w",
            encoding="utf-8"
        ) as f:

            f.write(html)


        # 이전 BLOCKED 파일 제거
        if os.path.exists(blocked_path):

            try:

                os.remove(blocked_path)

            except Exception as e:

                print(
                    f"⚠️ 기존 BLOCKED 파일 삭제 실패: {e}"
                )


        return normal_path


    # --------------------------------------------------------------------------
    # BLOCKED
    # --------------------------------------------------------------------------

    if status == "BLOCKED":

        with open(
            blocked_path,
            "w",
            encoding="utf-8"
        ) as f:

            f.write(html)


        # 이전 정상 HTML 제거
        if os.path.exists(normal_path):

            try:

                os.remove(normal_path)

            except Exception as e:

                print(
                    f"⚠️ 기존 정상 HTML 삭제 실패: {e}"
                )


        return blocked_path


    # --------------------------------------------------------------------------
    # FAIL / UNKNOWN
    # --------------------------------------------------------------------------

    with open(
        normal_path,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(html)


    return normal_path


# ================================================================================
# 21. Screenshot 저장
# ================================================================================

async def save_screenshot(
    page,
    asin,
    status
):

    try:

        filename = (
            f"{asin}_{status}.png"
        )

        path = os.path.join(
            SCREENSHOT_DIR,
            filename
        )


        await page.screenshot(
            path=path,
            full_page=False
        )


        return path


    except Exception as e:

        print(
            f"⚠️ Screenshot 저장 실패: {e}"
        )

        return None


# ================================================================================
# 22. 상품 1개 수집
# ================================================================================

async def collect_one(
    asin,
    product_url,
    attempt
):

    global page


    print(
        f"\n🌐 수집 시도 "
        f"{attempt}/2"
    )


    http_status = ""


    try:

        # ----------------------------------------------------------------------
        # Amazon URL
        # ----------------------------------------------------------------------

        response = await page.goto(

            product_url,

            wait_until="domcontentloaded",

            timeout=30000

        )


        if response:

            http_status = response.status

            print(
                f"HTTP status: "
                f"{http_status}"
            )


        # ----------------------------------------------------------------------
        # 페이지가 너무 빨리 끝나는 경우를 대비한 짧은 대기
        # ----------------------------------------------------------------------

        await asyncio.sleep(
            random.uniform(
                1.0,
                2.0
            )
        )


        # ----------------------------------------------------------------------
        # HTML 읽기
        # ----------------------------------------------------------------------

        html = await page.content()


        # ----------------------------------------------------------------------
        # HTML 분석
        # ----------------------------------------------------------------------

        status, reason = analyze_html(
            html
        )


        # ----------------------------------------------------------------------
        # BLOCKED
        # ----------------------------------------------------------------------

        if status == "BLOCKED":

            print(
                "🚫 Amazon BLOCKED"
            )

            print(
                f"   이유: {reason}"
            )


            saved_path = save_html(
                asin,
                html,
                "BLOCKED"
            )


            print(
                f"   저장: {saved_path}"
            )


            await save_screenshot(
                page,
                asin,
                "BLOCKED"
            )


            return {
                "ASIN": asin,
                "STATUS": "BLOCKED",
                "HTTP_STATUS": http_status,
                "REASON": reason,
                "HTML_FILE": saved_path,
                "수집시간": datetime.now().strftime(
                    "%Y-%m-%d %H:%M:%S"
                ),
            }


        # ----------------------------------------------------------------------
        # SUCCESS
        # ----------------------------------------------------------------------

        if status == "SUCCESS":

            print(
                "✅ 정상 Product HTML"
            )

            print(
                f"   이유: {reason}"
            )


            saved_path = save_html(
                asin,
                html,
                "SUCCESS"
            )


            print(
                f"   저장: {saved_path}"
            )


            return {
                "ASIN": asin,
                "STATUS": "SUCCESS",
                "HTTP_STATUS": http_status,
                "REASON": reason,
                "HTML_FILE": saved_path,
                "수집시간": datetime.now().strftime(
                    "%Y-%m-%d %H:%M:%S"
                ),
            }


        # ----------------------------------------------------------------------
        # FAIL / UNKNOWN
        # ----------------------------------------------------------------------

        print(
            f"⚠️ {status}"
        )

        print(
            f"   이유: {reason}"
        )


        saved_path = save_html(
            asin,
            html,
            status
        )


        return {
            "ASIN": asin,
            "STATUS": status,
            "HTTP_STATUS": http_status,
            "REASON": reason,
            "HTML_FILE": saved_path,
            "수집시간": datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            ),
        }


    except Exception as e:

        print(
            f"❌ 수집 오류: {e}"
        )


        return {
            "ASIN": asin,
            "STATUS": "ERROR",
            "HTTP_STATUS": http_status,
            "REASON": str(e),
            "HTML_FILE": "",
            "수집시간": datetime.now().strftime(
                "%Y-%m-%d %H:%M:%S"
            ),
        }


# ================================================================================
# 23. 수집 시작
# ================================================================================

print("\n" + "=" * 80)
print("🚀 Amazon HTML 수집 시작")
print("=" * 80)


MAX_ERROR_ATTEMPTS = 2

BROWSER_RESTART_INTERVAL = 40

WAIT_MIN = 2.0
WAIT_MAX = 5.0


# ------------------------------------------------------------------------------
# 결과 저장용
# ------------------------------------------------------------------------------

results = []


# ------------------------------------------------------------------------------
# 통계
# ------------------------------------------------------------------------------

stats = {

    "SUCCESS": 0,
    "BLOCKED": 0,
    "FAIL": 0,
    "UNKNOWN": 0,
    "ERROR": 0,

}


# ------------------------------------------------------------------------------
# 브라우저 시작
# ------------------------------------------------------------------------------

await start_browser()


# ================================================================================
# 24. 상품별 수집
# ================================================================================

for idx, row in enumerate(
    remaining_df.itertuples(index=False),
    start=1
):


    # --------------------------------------------------------------------------
    # ASIN
    # --------------------------------------------------------------------------

    asin = str(
        getattr(
            row,
            "ASIN"
        )
    ).strip().upper()


    # --------------------------------------------------------------------------
    # URL
    # --------------------------------------------------------------------------

    product_url = ""

    try:

        product_url = getattr(
            row,
            "상품URL"
        )

    except:

        product_url = ""


    if pd.isna(product_url):

        product_url = ""


    product_url = str(
        product_url
    ).strip()


    # URL이 없는 경우 기본 Amazon URL
    if not product_url:

        product_url = (
            f"https://www.amazon.com/dp/{asin}"
        )


    print("\n" + "=" * 80)

    print(
        f"[{idx}/{len(remaining_df)}] "
        f"ASIN: {asin}"
    )

    print(
        f"URL: {product_url}"
    )


    # --------------------------------------------------------------------------
    # 40개마다 브라우저 재시작
    # --------------------------------------------------------------------------

    if idx > 1 and (
        (idx - 1) % BROWSER_RESTART_INTERVAL == 0
    ):

        await start_browser()


    current_result = None


    # ==========================================================================
    # ASIN 수집
    # ==========================================================================

    for attempt in range(
        1,
        MAX_ERROR_ATTEMPTS + 1
    ):

        current_result = await collect_one(

            asin,
            product_url,
            attempt

        )


        status = current_result["STATUS"]


        # ----------------------------------------------------------------------
        # SUCCESS
        # ----------------------------------------------------------------------

        if status == "SUCCESS":

            stats["SUCCESS"] += 1

            break


        # ----------------------------------------------------------------------
        # BLOCKED
        #
        # 같은 ASIN에 대해 재시도하지 않음
        # ----------------------------------------------------------------------

        elif status == "BLOCKED":

            stats["BLOCKED"] += 1

            print(
                "   ⏭️ BLOCKED → "
                "재시도하지 않고 다음 ASIN으로 이동"
            )

            break


        # ----------------------------------------------------------------------
        # FAIL
        # ----------------------------------------------------------------------

        elif status == "FAIL":

            stats["FAIL"] += 1

            print(
                "   ⏭️ FAIL → "
                "현재 HTML 저장 후 다음 ASIN으로 이동"
            )

            break


        # ----------------------------------------------------------------------
        # UNKNOWN
        # ----------------------------------------------------------------------

        elif status == "UNKNOWN":

            stats["UNKNOWN"] += 1

            print(
                "   ⏭️ UNKNOWN → "
                "현재 HTML 저장 후 다음 ASIN으로 이동"
            )

            break


        # ----------------------------------------------------------------------
        # ERROR
        # ----------------------------------------------------------------------

        elif status == "ERROR":

            stats["ERROR"] += 1

            if attempt < MAX_ERROR_ATTEMPTS:

                print(
                    "   🔄 ERROR → "
                    "브라우저 재시작 후 재시도"
                )


                await start_browser()


                await asyncio.sleep(
                    2
                )


                continue


            else:

                print(
                    "   ❌ ERROR → "
                    "최대 재시도 횟수 초과"
                )

                break


    # ==========================================================================
    # Progress 저장
    #
    # 중요:
    #   기존 progress_df에 현재 결과 1건만 반영
    #   저장 후 반드시 progress_df를 갱신
    # ==========================================================================

    if current_result is not None:

        results.append(
            current_result
        )


        progress_df = save_progress(

            progress_df,

            current_result

        )


        print(
            f"💾 Progress 저장 "
            f"({len(progress_df):,}건)"
        )


    # ==========================================================================
    # 다음 상품 전 대기
    # ==========================================================================

    if idx < len(remaining_df):

        wait_sec = random.uniform(
            WAIT_MIN,
            WAIT_MAX
        )


        print(
            f"⏳ 다음 상품까지 "
            f"{wait_sec:.1f}초 대기"
        )


        await asyncio.sleep(
            wait_sec
        )


# ================================================================================
# 25. 브라우저 종료
# ================================================================================

print("\n" + "=" * 80)
print("🛑 브라우저 종료")
print("=" * 80)


await close_browser()


try:

    if playwright:

        await playwright.stop()

except:

    pass


# ================================================================================
# 26. 결과 DataFrame
# ================================================================================

results_df = pd.DataFrame(
    results
)


# ================================================================================
# 27. 수집결과 + 원본제품분석 결합
# ================================================================================

final_df = valid_df.copy()


# 기존 수집결과 제거
for col in [

    "STATUS",
    "HTTP_STATUS",
    "REASON",
    "HTML_FILE",
    "수집시간",

]:

    if col in final_df.columns:

        final_df.drop(
            columns=[col],
            inplace=True
        )


# 수집 결과 병합
if not results_df.empty:

    final_df = final_df.merge(

        results_df[
            [
                "ASIN",
                "STATUS",
                "HTTP_STATUS",
                "REASON",
                "HTML_FILE",
                "수집시간",
            ]
        ],

        on="ASIN",

        how="left"

    )


# ================================================================================
# 28. Excel 저장
# ================================================================================

print("\n" + "=" * 80)
print("📊 Excel 결과 저장")
print("=" * 80)


with pd.ExcelWriter(
    RESULT_FILE,
    engine="openpyxl"
) as writer:

    final_df.to_excel(
        writer,
        sheet_name="수집결과",
        index=False
    )


    valid_df.to_excel(
        writer,
        sheet_name="원본제품분석",
        index=False
    )


print(
    f"✅ 결과 Excel 저장 완료:"
)

print(
    RESULT_FILE
)


# ================================================================================
# 29. 최종 통계
# ================================================================================

print("\n" + "=" * 80)
print("📊 최종 수집 통계")
print("=" * 80)


print(
    f"전체 수집 대상 : "
    f"{len(remaining_df):,}"
)

print(
    f"SUCCESS        : "
    f"{stats['SUCCESS']:,}"
)

print(
    f"BLOCKED        : "
    f"{stats['BLOCKED']:,}"
)

print(
    f"FAIL           : "
    f"{stats['FAIL']:,}"
)

print(
    f"UNKNOWN        : "
    f"{stats['UNKNOWN']:,}"
)

print(
    f"ERROR          : "
    f"{stats['ERROR']:,}"
)


# ================================================================================
# 30. 실제 Drive HTML 파일 통계
# ================================================================================

normal_html_count = 0
blocked_html_count = 0


for filename in os.listdir(
    HTML_DIR
):

    if filename.endswith(
        "_BLOCKED.html"
    ):

        blocked_html_count += 1

    elif filename.endswith(
        ".html"
    ):

        normal_html_count += 1


print("\n" + "=" * 80)
print("📁 Drive HTML 파일 현황")
print("=" * 80)


print(
    f"정상 HTML       : "
    f"{normal_html_count:,}개"
)

print(
    f"BLOCKED HTML    : "
    f"{blocked_html_count:,}개"
)


# ================================================================================
# 31. ZIP 생성
# ================================================================================

print("\n" + "=" * 80)
print("📦 HTML ZIP 생성")
print("=" * 80)


timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


zip_path = (
    f"{ZIP_BASE}_{timestamp}.zip"
)


html_files = [

    filename

    for filename in os.listdir(
        HTML_DIR
    )

    if filename.endswith(
        ".html"
    )

]


with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:

    for filename in html_files:

        file_path = os.path.join(
            HTML_DIR,
            filename
        )

        zf.write(
            file_path,
            arcname=filename
        )


print(
    f"✅ ZIP 생성 완료:"
)

print(
    zip_path
)

print(
    f"📦 ZIP 포함 HTML: "
    f"{len(html_files):,}개"
)


# ================================================================================
# 32. 최종 완료
# ================================================================================

print("\n" + "=" * 80)
print("🎉 Amazon HTML 수집 완료")
print("=" * 80)

print(
    f"📄 Excel : {RESULT_FILE}"
)

print(
    f"📦 ZIP   : {zip_path}"
)

print(
    f"📁 HTML  : {HTML_DIR}"
)

print(
    f"📋 Progress : {PROGRESS_FILE}"
)


# ================================================================================
# 33. 저장된 Amazon HTML → 상품 상세정보 추출
#
# 추출 대상
#   - ASIN
#   - 브랜드
#   - 상품명
#   - 상품URL
#   - Item Form
#   - Product Benefits
#   - Scent Name
#   - Skin Type
#   - Active Ingredients
#   - Skin Tone
#   - Included Components
#   - UPC
#   - Global Trade Identification Number
#   - Manufacturer
#   - Model Number
#   - Manufacturer Part Number
#   - Best Sellers Rank
#   - Customer Reviews
#   - Rating
#
# 기존 HTML_DIR에 저장된 HTML을 다시 읽음
# ================================================================================

from bs4 import BeautifulSoup


print("\n" + "=" * 80)
print("🔎 Amazon 상품 상세정보 추출 시작")
print("=" * 80)


# ================================================================================
# 33-1. 텍스트 정리 함수
# ================================================================================

def clean_amazon_text(value):

    if value is None:
        return ""

    value = str(value)

    # HTML entity / 공백 정리
    value = re.sub(
        r"\s+",
        " ",
        value
    )

    return value.strip()


# ================================================================================
# 33-2. 상품 상세정보 추출 함수
# ================================================================================

def extract_amazon_product_info(
    html,
    asin_from_filename=""
):

    result = {

        "ASIN": asin_from_filename,

        "Brand": "",
        "Product Name": "",
        "Product URL": "",

        "Item Form": "",
        "Product Benefits": "",
        "Scent Name": "",
        "Skin Type": "",
        "Active Ingredients": "",
        "Skin Tone": "",

        "Included Components": "",
        "UPC": "",
        "Global Trade Identification Number": "",
        "Manufacturer": "",
        "Model Number": "",
        "Manufacturer Part Number": "",

        "Best Sellers Rank": "",
        "Customer Reviews": "",
        "Rating": "",

    }


    if not html:

        return result


    soup = BeautifulSoup(
        html,
        "html.parser"
    )


    # =============================================================================
    # ① ASIN
    # =============================================================================

    # 파일명 ASIN 우선
    if not result["ASIN"]:

        asin_candidates = [

            soup.select_one(
                "#ASIN"
            ),

            soup.select_one(
                'input[name="ASIN"]'
            ),

            soup.select_one(
                '[data-asin]'
            ),

        ]


        for tag in asin_candidates:

            if not tag:
                continue


            if tag.name == "input":

                value = tag.get(
                    "value",
                    ""
                )

            else:

                value = tag.get(
                    "value",
                    ""
                ) or tag.get(
                    "data-asin",
                    ""
                )


            value = clean_amazon_text(
                value
            )


            if re.fullmatch(
                r"[A-Z0-9]{10}",
                value.upper()
            ):

                result["ASIN"] = value.upper()

                break


    # =============================================================================
    # ② Product Name
    # =============================================================================

    product_name_selectors = [

        "#productTitle",

        "h1#title",

        "span#productTitle",

        "h1.a-size-large",

    ]


    for selector in product_name_selectors:

        tag = soup.select_one(
            selector
        )

        if tag:

            value = clean_amazon_text(
                tag.get_text(
                    " ",
                    strip=True
                )
            )

            if value:

                result["Product Name"] = value

                break


    # =============================================================================
    # ③ Product URL
    # =============================================================================

    canonical = soup.select_one(
        'link[rel="canonical"]'
    )


    if canonical:

        result["Product URL"] = clean_amazon_text(
            canonical.get(
                "href",
                ""
            )
        )


    # canonical이 없으면 og:url
    if not result["Product URL"]:

        og_url = soup.select_one(
            'meta[property="og:url"]'
        )

        if og_url:

            result["Product URL"] = clean_amazon_text(
                og_url.get(
                    "content",
                    ""
                )
            )


    # =============================================================================
    # ④ Amazon Product Specifications
    #
    # 핵심:
    # table.a-keyvalue.prodDetTable
    # 안의
    # <th> = 항목명
    # <td> = 값
    # =============================================================================

    specs = {}


    tables = soup.select(
        "table.a-keyvalue.prodDetTable"
    )


    for table in tables:

        rows = table.select(
            "tr"
        )


        for tr in rows:

            th = tr.select_one(
                "th"
            )

            td = tr.select_one(
                "td"
            )


            if not th or not td:

                continue


            key = clean_amazon_text(
                th.get_text(
                    " ",
                    strip=True
                )
            )


            value = clean_amazon_text(
                td.get_text(
                    " ",
                    strip=True
                )
            )


            if not key:

                continue


            # 같은 항목이 중복될 경우
            # 값이 비어 있지 않은 것 우선
            if (
                key not in specs
                or not specs[key]
            ):

                specs[key] = value


    # =============================================================================
    # ⑤ Brand
    # =============================================================================

    result["Brand"] = specs.get(
        "Brand Name",
        ""
    )


    # 혹시 Brand Name이 없는 Amazon 페이지
    # 다른 위치에서 보조 추출
    if not result["Brand"]:

        brand_selectors = [

            "#bylineInfo",

            "a#bylineInfo",

            ".po-brand",

        ]


        for selector in brand_selectors:

            tag = soup.select_one(
                selector
            )


            if tag:

                value = clean_amazon_text(
                    tag.get_text(
                        " ",
                        strip=True
                    )
                )


                value = re.sub(
                    r"^(Visit the|Brand:?)\s*",
                    "",
                    value,
                    flags=re.I
                ).strip()


                if value:

                    result["Brand"] = value

                    break


    # =============================================================================
    # ⑥ Product Specifications 매핑
    # =============================================================================

    field_mapping = {

        "Item Form":
            "Item Form",

        "Product Benefits":
            "Product Benefits",

        "Scent Name":
            "Scent Name",

        "Skin Type":
            "Skin Type",

        "Active Ingredients":
            "Active Ingredients",

        "Skin Tone":
            "Skin Tone",

        "Included Components":
            "Included Components",

        "UPC":
            "UPC",

        "Global Trade Identification Number":
            "Global Trade Identification Number",

        "Manufacturer":
            "Manufacturer",

        "Model Number":
            "Model Number",

        "Manufacturer Part Number":
            "Manufacturer Part Number",

        "Best Sellers Rank":
            "Best Sellers Rank",

    }


    for output_col, amazon_key in field_mapping.items():

        result[output_col] = specs.get(
            amazon_key,
            ""
        )


    # =============================================================================
    # ⑦ Best Sellers Rank
    #
    # ul/li 구조인 경우 get_text로 전체 순위를 가져옴
    # =============================================================================

    if not result["Best Sellers Rank"]:

        for tr in soup.select(
            "table.a-keyvalue.prodDetTable tr"
        ):

            th = tr.select_one(
                "th"
            )


            if not th:
                continue


            key = clean_amazon_text(
                th.get_text(
                    " ",
                    strip=True
                )
            )


            if key.lower() == "best sellers rank":

                td = tr.select_one(
                    "td"
                )

                if td:

                    result["Best Sellers Rank"] = clean_amazon_text(
                        td.get_text(
                            " ",
                            strip=True
                        )
                    )

                    break


    # =============================================================================
    # ⑧ Customer Reviews
    # =============================================================================

    review_tag = soup.select_one(
        "#acrCustomerReviewText"
    )


    if review_tag:

        result["Customer Reviews"] = clean_amazon_text(
            review_tag.get_text(
                " ",
                strip=True
            )
        )


    # =============================================================================
    # ⑨ Rating
    # =============================================================================

    rating_selectors = [

        "#acrPopover",

        'span[data-hook="rating-out-of-text"]',

        'i[data-hook="average-star-rating"]',

    ]


    for selector in rating_selectors:

        tag = soup.select_one(
            selector
        )


        if not tag:

            continue


        rating = ""


        # title
        rating = tag.get(
            "title",
            ""
        )


        # aria-label
        if not rating:

            rating = tag.get(
                "aria-label",
                ""
            )


        # 텍스트
        if not rating:

            rating = tag.get_text(
                " ",
                strip=True
            )


        rating = clean_amazon_text(
            rating
        )


        if rating:

            result["Rating"] = rating

            break


    # =============================================================================
    # ⑩ 최종 ASIN fallback
    # =============================================================================

    if not result["ASIN"]:

        asin_match = re.search(
            r"\b([A-Z0-9]{10})\b",
            html,
            flags=re.I
        )


        if asin_match:

            result["ASIN"] = asin_match.group(1).upper()


    return result


# ================================================================================
# 33-3. HTML 파일 전체 읽기
# ================================================================================

html_filenames = [

    filename

    for filename in os.listdir(
        HTML_DIR
    )

    if filename.lower().endswith(
        ".html"
    )

    and not filename.upper().endswith(
        "_BLOCKED.HTML"
    )

]


print(
    f"📄 분석 대상 정상 HTML: "
    f"{len(html_filenames):,}개"
)


# ================================================================================
# 33-4. HTML별 추출
# ================================================================================

detail_results = []


for file_idx, filename in enumerate(
    html_filenames,
    start=1
):


    file_path = os.path.join(
        HTML_DIR,
        filename
    )


    # 파일명에서 ASIN 추출
    asin_match = re.match(
        r"^([A-Z0-9]{10})\.html$",
        filename,
        flags=re.I
    )


    if asin_match:

        asin = asin_match.group(1).upper()

    else:

        asin = ""


    try:

        with open(
            file_path,
            "r",
            encoding="utf-8",
            errors="ignore"
        ) as f:

            html = f.read()


        info = extract_amazon_product_info(
            html,
            asin_from_filename=asin
        )


        # 원본 HTML 파일명
        info["HTML_FILE"] = file_path


        detail_results.append(
            info
        )


        print(
            f"[{file_idx}/{len(html_filenames)}] "
            f"✅ {info['ASIN']} | "
            f"{info['Brand']} | "
            f"{info['Product Name'][:80]}"
        )


    except Exception as e:

        print(
            f"[{file_idx}/{len(html_filenames)}] "
            f"❌ {filename} | {e}"
        )


# ================================================================================
# 33-5. DataFrame
# ================================================================================

product_detail_df = pd.DataFrame(
    detail_results
)


print("\n" + "=" * 80)
print("📊 상품 상세정보 추출 결과")
print("=" * 80)


print(
    f"추출 행 수: "
    f"{len(product_detail_df):,}"
)


if not product_detail_df.empty:

    print(
        f"브랜드 추출 성공: "
        f"{(product_detail_df['Brand'].astype(str).str.strip() != '').sum():,}"
    )


    print(
        f"상품명 추출 성공: "
        f"{(product_detail_df['Product Name'].astype(str).str.strip() != '').sum():,}"
    )


    print(
        f"Active Ingredients 추출 성공: "
        f"{(product_detail_df['Active Ingredients'].astype(str).str.strip() != '').sum():,}"
    )


    print("\n📋 컬럼:")

    print(
        product_detail_df.columns.tolist()
    )


# ================================================================================
# 34. 기존 Excel에 상품 상세정보 Sheet 추가
# ================================================================================

print("\n" + "=" * 80)
print("📊 상품 상세정보 Excel 저장")
print("=" * 80)


with pd.ExcelWriter(
    RESULT_FILE,
    engine="openpyxl",
    mode="a",
    if_sheet_exists="replace"
) as writer:

    product_detail_df.to_excel(
        writer,
        sheet_name="상품상세정보",
        index=False
    )


print(
    f"✅ 상품상세정보 Sheet 저장 완료"
)

print(
    RESULT_FILE
)


# ================================================================================
# 35. 상세정보 추출 결과 미리보기
# ================================================================================

if not product_detail_df.empty:

    preview_columns = [

        "ASIN",
        "Brand",
        "Product Name",
        "Item Form",
        "Product Benefits",
        "Active Ingredients",
        "Skin Type",
        "Scent Name",
        "Skin Tone",
        "Included Components",
        "Manufacturer",
        "Model Number",
        "Best Sellers Rank",
        "Customer Reviews",
        "Rating",

    ]


    preview_columns = [

        col

        for col in preview_columns

        if col in product_detail_df.columns

    ]


    print("\n📋 상품 상세정보 미리보기")

    display(
        product_detail_df[
            preview_columns
        ].head(10)
    )


# ================================================================================
# 36. Excel 다운로드
# ================================================================================

print("\n" + "=" * 80)
print("📥 Excel 다운로드")
print("=" * 80)


if os.path.exists(
    RESULT_FILE
):

    files.download(
        RESULT_FILE
    )


    print(
        "✅ Excel 다운로드 시작"
    )

else:

    print(
        "❌ 결과 Excel 파일을 찾을 수 없습니다."
    )

In [ ]:
# ============================================================
# Amazon 기본파일 + 상세페이지 정보 ASIN 병합
# ============================================================
# 기준:
#   1. 기본파일 = MASTER
#   2. 상세페이지 = 보완 데이터
#   3. ASIN으로 매칭
#   4. 기본파일 값이 있으면 유지
#   5. 기본파일 빈칸만 상세페이지 값으로 채움
#   6. 기본파일에 없는 ASIN은 상세페이지에서도 제거
#   7. 상세페이지 중복 ASIN은 하나로 통합
# ============================================================

import pandas as pd
import numpy as np
import os
import re
from google.colab import files
from openpyxl import load_workbook


# ============================================================
# 1. Excel 파일 업로드
# ============================================================

print("=" * 70)
print("📂 기본파일 Excel을 업로드하세요.")
print("=" * 70)

uploaded_base = files.upload()

base_filename = list(uploaded_base.keys())[0]

print(f"\n✅ 기본파일: {base_filename}")


print("\n" + "=" * 70)
print("📂 상세페이지 Excel을 업로드하세요.")
print("=" * 70)

uploaded_detail = files.upload()

detail_filename = list(uploaded_detail.keys())[0]

print(f"\n✅ 상세페이지 파일: {detail_filename}")


# ============================================================
# 2. Excel 시트 확인
# ============================================================

print("\n" + "=" * 70)
print("📋 파일 시트 확인")
print("=" * 70)

base_xls = pd.ExcelFile(base_filename)
detail_xls = pd.ExcelFile(detail_filename)

print("\n[기본파일]")
for s in base_xls.sheet_names:
    print(" -", s)

print("\n[상세페이지 파일]")
for s in detail_xls.sheet_names:
    print(" -", s)


# ============================================================
# 3. 기본파일 읽기
# ============================================================

# 기본파일의 제품분석 시트 사용
if "제품분석" in base_xls.sheet_names:
    base_df = pd.read_excel(
        base_filename,
        sheet_name="제품분석"
    )
    base_sheet_name = "제품분석"
else:
    # 제품분석이 없으면 첫 번째 시트 사용
    base_sheet_name = base_xls.sheet_names[0]

    print(
        f"\n⚠️ 기본파일에 '제품분석' 시트가 없어 "
        f"'{base_sheet_name}' 시트를 사용합니다."
    )

    base_df = pd.read_excel(
        base_filename,
        sheet_name=base_sheet_name
    )


# ============================================================
# 4. 상세페이지 정보 읽기
# ============================================================

if "상품상세정보" not in detail_xls.sheet_names:
    raise ValueError(
        "\n❌ 상세페이지 파일에 '상품상세정보' 시트가 없습니다.\n"
        f"현재 시트: {detail_xls.sheet_names}"
    )

detail_df = pd.read_excel(
    detail_filename,
    sheet_name="상품상세정보"
)


# ============================================================
# 5. 컬럼명 정리
# ============================================================

base_df.columns = [
    str(c).strip()
    for c in base_df.columns
]

detail_df.columns = [
    str(c).strip()
    for c in detail_df.columns
]


# ============================================================
# 6. ASIN 컬럼 찾기
# ============================================================

def find_asin_column(df, filename=""):
    """
    ASIN 컬럼을 최대한 유연하게 찾음.
    """

    # 정확한 컬럼명 우선
    candidates = [
        "ASIN",
        "asin",
        "Asin",
        "상품ASIN",
        "상품 Asin",
        "ASIN번호",
        "ASIN 번호"
    ]

    for col in candidates:
        if col in df.columns:
            return col

    # 컬럼명에 ASIN이 포함되어 있는 경우
    for col in df.columns:
        col_str = str(col).strip().lower()

        if "asin" in col_str:
            return col

    raise ValueError(
        f"\n❌ ASIN 컬럼을 찾을 수 없습니다: {filename}\n"
        f"현재 컬럼:\n{list(df.columns)}"
    )


base_asin_col = find_asin_column(
    base_df,
    base_filename
)

detail_asin_col = find_asin_column(
    detail_df,
    detail_filename
)

print("\n" + "=" * 70)
print("🔎 ASIN 컬럼")
print("=" * 70)

print("기본파일 :", base_asin_col)
print("상세파일 :", detail_asin_col)


# ============================================================
# 7. ASIN 정규화 함수
# ============================================================

def normalize_asin(value):
    """
    ASIN을 비교용 문자열로 정규화.
    """

    if pd.isna(value):
        return ""

    value = str(value).strip()

    # Excel에서 숫자처럼 들어온 경우
    if value.endswith(".0"):
        value = value[:-2]

    value = value.upper()

    # 앞뒤 공백 제거
    value = value.strip()

    return value


# 비교용 ASIN 컬럼 생성
base_df["_ASIN_KEY"] = base_df[base_asin_col].apply(
    normalize_asin
)

detail_df["_ASIN_KEY"] = detail_df[detail_asin_col].apply(
    normalize_asin
)


# ============================================================
# 8. 빈 ASIN 제거
# ============================================================

base_before = len(base_df)
detail_before = len(detail_df)

base_df = base_df[
    base_df["_ASIN_KEY"] != ""
].copy()

detail_df = detail_df[
    detail_df["_ASIN_KEY"] != ""
].copy()

print("\n빈 ASIN 제거")
print(f"기본파일: {base_before} → {len(base_df)}")
print(f"상세파일: {detail_before} → {len(detail_df)}")


# ============================================================
# 9. 기본파일 ASIN 집합 생성
# ============================================================

base_asins = set(
    base_df["_ASIN_KEY"]
)

detail_asins_before = set(
    detail_df["_ASIN_KEY"]
)


# ============================================================
# 10. ★ 기본파일에 없는 ASIN 상세페이지에서 제거
# ============================================================

detail_only_asins = (
    detail_asins_before - base_asins
)

matched_asins = (
    detail_asins_before & base_asins
)

print("\n" + "=" * 70)
print("🔗 ASIN 매칭 결과")
print("=" * 70)

print(f"기본파일 ASIN 수             : {len(base_asins):,}")
print(f"상세페이지 ASIN 수           : {len(detail_asins_before):,}")
print(f"양쪽에 존재하는 ASIN         : {len(matched_asins):,}")
print(f"상세페이지에만 존재하는 ASIN : {len(detail_only_asins):,}")


# ★ 중요
# 기본파일에 없는 제품은 상세페이지에서도 완전히 제거
detail_df = detail_df[
    detail_df["_ASIN_KEY"].isin(base_asins)
].copy()

print(
    f"\n✅ 상세페이지 데이터 필터링 완료: "
    f"{len(detail_df):,}행"
)


# ============================================================
# 11. 상세페이지 중복 ASIN 통합
# ============================================================

print("\n" + "=" * 70)
print("🔄 상세페이지 중복 ASIN 통합")
print("=" * 70)

detail_duplicate_count = (
    detail_df["_ASIN_KEY"]
    .duplicated(keep=False)
    .sum()
)

detail_unique_before_merge = (
    detail_df["_ASIN_KEY"].nunique()
)

print(
    f"중복 ASIN 행 수: {detail_duplicate_count:,}"
)

print(
    f"통합 전 ASIN 수: {detail_unique_before_merge:,}"
)


# ============================================================
# 12. 여러 행의 값을 합치는 함수
# ============================================================

def merge_cell_values(series):
    """
    여러 셀 값을 하나로 합침.

    우선순위:
    - 빈 값 무시
    - 중복 값 제거
    - 기존 순서 유지
    - HTML_FILE 계열은 파일명을 ; 로 연결
    """

    values = []

    for value in series:

        if pd.isna(value):
            continue

        value = str(value).strip()

        if not value:
            continue

        # 중복 제거
        if value not in values:
            values.append(value)

    if not values:
        return np.nan

    return " ; ".join(values)


# ============================================================
# 13. 상세페이지 ASIN별 통합
# ============================================================

# ASIN을 제외한 실제 데이터 컬럼
detail_data_cols = [
    c for c in detail_df.columns
    if c != "_ASIN_KEY"
]


detail_merged = (
    detail_df
    .groupby("_ASIN_KEY", sort=False)
    [detail_data_cols]
    .agg(merge_cell_values)
    .reset_index()
)


print(
    f"통합 후 ASIN 수: "
    f"{detail_merged['_ASIN_KEY'].nunique():,}"
)


# ============================================================
# 14. 상세페이지 Lookup 생성
# ============================================================

detail_lookup = (
    detail_merged
    .set_index("_ASIN_KEY")
    .to_dict("index")
)


# ============================================================
# 15. 기본파일에 없는 상세 컬럼 확인
# ============================================================

base_original_cols = [
    c for c in base_df.columns
    if c != "_ASIN_KEY"
]

detail_original_cols = [
    c for c in detail_merged.columns
    if c != "_ASIN_KEY"
]


# 상세페이지에만 있는 컬럼
new_detail_cols = [
    c for c in detail_original_cols
    if c not in base_original_cols
]


print("\n" + "=" * 70)
print("📋 컬럼 분석")
print("=" * 70)

print(
    f"기본파일 컬럼 수        : "
    f"{len(base_original_cols)}"
)

print(
    f"상세페이지 컬럼 수      : "
    f"{len(detail_original_cols)}"
)

print(
    f"새로 추가될 컬럼 수     : "
    f"{len(new_detail_cols)}"
)


if new_detail_cols:
    print("\n🆕 추가되는 상세페이지 컬럼:")

    for col in new_detail_cols:
        print(" -", col)


# ============================================================
# 16. 새 상세페이지 컬럼 생성
# ============================================================

for col in new_detail_cols:
    base_df[col] = np.nan


# ============================================================
# 17. 기본파일 값은 유지 + 빈칸만 상세정보로 보완
# ============================================================

print("\n" + "=" * 70)
print("🔄 데이터 병합")
print("=" * 70)


filled_count = 0
already_filled_count = 0
no_detail_count = 0


for idx, row in base_df.iterrows():

    asin_key = row["_ASIN_KEY"]

    # 상세정보가 없는 기본상품
    if asin_key not in detail_lookup:
        no_detail_count += 1
        continue

    detail_record = detail_lookup[asin_key]

    for col, detail_value in detail_record.items():

        # 빈 상세정보는 무시
        if pd.isna(detail_value):
            continue

        detail_value = str(detail_value).strip()

        if not detail_value:
            continue

        # 현재 기본파일 값
        current_value = base_df.at[idx, col]

        # ----------------------------------------------------
        # 기본파일 값이 비어 있을 때만 채움
        # ----------------------------------------------------

        is_blank = (
            pd.isna(current_value)
            or str(current_value).strip() == ""
        )

        if is_blank:

            base_df.at[idx, col] = detail_value

            filled_count += 1

        else:

            # 기존 값은 유지
            already_filled_count += 1


print(f"\n✅ 빈칸 보완 완료")
print(f"상세정보로 새롭게 채운 셀 : {filled_count:,}")
print(f"기존 값이 있어 유지한 셀   : {already_filled_count:,}")
print(f"상세정보가 없는 상품       : {no_detail_count:,}")


# ============================================================
# 18. 내부용 _ASIN_KEY 제거
# ============================================================

if "_ASIN_KEY" in base_df.columns:
    base_df = base_df.drop(
        columns=["_ASIN_KEY"]
    )

if "_ASIN_KEY" in detail_merged.columns:
    detail_merged = detail_merged.drop(
        columns=["_ASIN_KEY"]
    )


# ============================================================
# 19. 최종 상세페이지 데이터도 기본파일 ASIN만 유지
# ============================================================

# 안전장치
base_final_asins = set(
    base_df[base_asin_col]
    .apply(normalize_asin)
)

detail_merged["_FINAL_ASIN_KEY"] = (
    detail_merged[detail_asin_col]
    .apply(normalize_asin)
)

detail_merged = detail_merged[
    detail_merged["_FINAL_ASIN_KEY"].isin(
        base_final_asins
    )
].copy()

detail_merged = detail_merged.drop(
    columns=["_FINAL_ASIN_KEY"]
)


# ============================================================
# 20. 병합 로그 생성
# ============================================================

log_rows = []

log_rows.append({
    "항목": "기본파일명",
    "값": base_filename
})

log_rows.append({
    "항목": "상세페이지 파일명",
    "값": detail_filename
})

log_rows.append({
    "항목": "기본파일 제품 수",
    "값": len(base_df)
})

log_rows.append({
    "항목": "기본파일 고유 ASIN 수",
    "값": len(base_final_asins)
})

log_rows.append({
    "항목": "상세페이지 원본 행 수",
    "값": detail_before
})

log_rows.append({
    "항목": "상세페이지 원본 고유 ASIN 수",
    "값": len(detail_asins_before)
})

log_rows.append({
    "항목": "매칭 ASIN 수",
    "값": len(matched_asins)
})

log_rows.append({
    "항목": "상세페이지에만 있던 ASIN 수",
    "값": len(detail_only_asins)
})

log_rows.append({
    "항목": "상세페이지 필터링 후 행 수",
    "값": len(detail_df)
})

log_rows.append({
    "항목": "상세페이지 통합 후 ASIN 수",
    "값": len(detail_merged)
})

log_rows.append({
    "항목": "상세정보로 채운 셀 수",
    "값": filled_count
})

log_rows.append({
    "항목": "기존 값 유지 셀 수",
    "값": already_filled_count
})

log_rows.append({
    "항목": "상세정보 없는 기본상품 수",
    "값": no_detail_count
})

log_df = pd.DataFrame(log_rows)


# ============================================================
# 21. 결과 파일명
# ============================================================

base_name_without_ext = os.path.splitext(
    os.path.basename(base_filename)
)[0]

output_filename = (
    f"{base_name_without_ext}_상세정보병합.xlsx"
)


# ============================================================
# 22. Excel 저장
# ============================================================

print("\n" + "=" * 70)
print("💾 Excel 저장")
print("=" * 70)

with pd.ExcelWriter(
    output_filename,
    engine="openpyxl"
) as writer:

    # --------------------------------------------
    # 제품분석
    # --------------------------------------------
    base_df.to_excel(
        writer,
        sheet_name="제품분석",
        index=False
    )

    # --------------------------------------------
    # 상품상세정보
    # --------------------------------------------
    detail_merged.to_excel(
        writer,
        sheet_name="상품상세정보",
        index=False
    )

    # --------------------------------------------
    # 로그
    # --------------------------------------------
    log_df.to_excel(
        writer,
        sheet_name="MERGE_로그",
        index=False
    )


# ============================================================
# 23. Excel 컬럼 너비 조정
# ============================================================

wb = load_workbook(output_filename)

for ws in wb.worksheets:

    for column_cells in ws.columns:

        max_length = 0

        column_letter = column_cells[0].column_letter

        for cell in column_cells:

            try:
                cell_length = len(
                    str(cell.value)
                )

                if cell_length > max_length:
                    max_length = cell_length

            except:
                pass

        # 너무 넓어지지 않도록 제한
        adjusted_width = min(
            max(max_length + 2, 10),
            60
        )

        ws.column_dimensions[
            column_letter
        ].width = adjusted_width


# 첫 행 고정
for ws in wb.worksheets:
    ws.freeze_panes = "A2"


wb.save(output_filename)


# ============================================================
# 24. 최종 검증
# ============================================================

print("\n" + "=" * 70)
print("🔍 최종 검증")
print("=" * 70)

final_xls = pd.ExcelFile(
    output_filename
)

final_product_df = pd.read_excel(
    output_filename,
    sheet_name="제품분석"
)

final_detail_df = pd.read_excel(
    output_filename,
    sheet_name="상품상세정보"
)


final_product_asins = set(
    final_product_df[base_asin_col]
    .apply(normalize_asin)
)

final_detail_asins = set(
    final_detail_df[detail_asin_col]
    .apply(normalize_asin)
)


# 상세페이지에만 존재하는 ASIN
remaining_extra_asins = (
    final_detail_asins - final_product_asins
)


print(
    f"최종 제품분석 행 수       : "
    f"{len(final_product_df):,}"
)

print(
    f"최종 제품분석 ASIN 수     : "
    f"{len(final_product_asins):,}"
)

print(
    f"최종 상품상세정보 행 수   : "
    f"{len(final_detail_df):,}"
)

print(
    f"최종 상품상세정보 ASIN 수 : "
    f"{len(final_detail_asins):,}"
)

print(
    f"기본파일에 없는 ASIN      : "
    f"{len(remaining_extra_asins):,}"
)


if len(remaining_extra_asins) == 0:
    print(
        "\n✅ 검증 성공!"
    )
    print(
        "   상세페이지에 기본파일에 없는 ASIN이 없습니다."
    )
else:
    print(
        "\n⚠️ 검증 필요:"
    )

    for asin in list(remaining_extra_asins)[:20]:
        print(" -", asin)


# ============================================================
# 25. 최종 다운로드
# ============================================================

print("\n" + "=" * 70)
print("📥 최종 파일 다운로드")
print("=" * 70)

print(
    f"\n파일명: {output_filename}"
)

files.download(
    output_filename
)

print("\n🎉 완료!")